# t0207: Elsys tpc5 + NI npz → Labquake Explorer HDF5

Preprocessing for an experiment recorded on two systems.

* **Elsys TranAX** (`.tpc5`, ECR dual mode): block 1 is a 2 kHz continuous record of the
  whole run, blocks 2…N are 2 MHz records around each PZT trigger (0.1 s before, 0.5 s after).
* **National Instruments USB-6451** (`.npz`, run5 only): the mechanical channels recorded
  continuously at 500 kHz on the NI clock, with `trigger_sample_index` marking the sample at
  which the first Elsys trigger pulse arrived (the Elsys only emits that pulse once).

Runs 1–4 had every sensor on the Elsys and use the tpc5-only recipe of `t0211_tpc5.ipynb`.
In run5 the mechanical sensors were moved to the NI logger (the Elsys channels A5–A7 and
B1–B8 read 0 V), so that run's time history is the NI record decimated to 2 kHz, on the NI
clock, with the Elsys PZTs interpolated onto it; both files stay referenced for event
extraction, the Elsys one shifted by `elsys.time_offset`.

Run sheet (2026-08-20, 21 °C, 57 % RH, flow rate 8 ml/min, operators 朱展穎、張德權):

| Elsys channel | sensor        | input range (Elsys setting) |
|---------------|---------------|-----------------------------|
| A1–A4         | PZT           | ±0.25 V                     |
| A5, A6        | pressure      | 0–10 V                      |
| A7            | LVDT          | −6 to 14 V                  |
| B1–B8         | eddy current  | ±25 V                       |

NI channels (research log "[T207] NI-6451 DAQ Trial Run", 2026-08-20): AI-0 normal-stress
voltage, AI-1 shear-stress voltage, AI-2 LVDT, AI-3 an eddy-current sensor with a 9 V dry cell
in series as an offset trial (it reads about 9 V above the others), AI-4 to AI-10 the usual
eddy-current sensors; ±10 V input range, which is narrower than the eddy sensors' −25 to 0 V
output (run5 reaches −9.8 V without clipping). Remark on the run sheet: after run2 block 20
the E02 slip channel is suspect.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib widget

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))   # repository root, if not pip-installed
from labquake_explorer.data.data_manager import DataManager
from labquake_explorer.data.sources import Tpc5Source, NINpzSource, open_source
from labquake_explorer.data.channels import get_channel, aligned_fields
from labquake_explorer.preprocessing import (
    Calibration, EddySlip, Friction, pressure_transducers, experiment,
    run_from_tpc5, run_from_tpc5_ni, slip_step_table,
)

## Paths

NI files are paired with tpc5 files by the `runN` token in their names.

In [ ]:
import re

EXPERIMENT = 't0207'
EXPERIMENT_DIR = Path('/Users/hueyke/Library/CloudStorage/SynologyDrive-data/Experiment/T0207')
DATA_DIR = EXPERIMENT_DIR / f'{EXPERIMENT}.exp' / 'data'
OUTPUT_FILE = EXPERIMENT_DIR / f'{EXPERIMENT}.h5'   # raw-file paths are stored relative to it

TPC5_FILES = sorted(DATA_DIR.glob(f'{EXPERIMENT}_*.tpc5'))
NI_FILES = {m.group(1): p for p in DATA_DIR.glob('*.npz') for m in [re.search(r'(run\d+)', p.name)] if m}
run_of = lambda p: re.search(r'(run\d+)', p.stem).group(1)

pd.DataFrame([{'run': run_of(p), 'tpc5': p.name, 'tpc5 (MB)': round(p.stat().st_size / 1e6),
               'ni npz': NI_FILES[run_of(p)].name if run_of(p) in NI_FILES else '',
               'ni (MB)': round(NI_FILES[run_of(p)].stat().st_size / 1e6) if run_of(p) in NI_FILES else ''}
              for p in TPC5_FILES])

## What is in the files

The NI record and the tpc5 block table of run5 (metadata only).

In [ ]:
ni = NINpzSource.open(NI_FILES['run5'])
display(pd.Series({k: v for k, v in ni.record.as_dict().items() if k != 'member_offsets'}))
src5 = Tpc5Source.open(DATA_DIR / 't0207_04_16MPa_run5.tpc5')
display(pd.DataFrame([b.as_dict() for b in src5.blocks]))

## Channel roles and calibration

`ELSYS_CHANNELS` and `NI_CHANNELS` map each recorder's channel label to the field stored in the run (volts).
`CALIBRATION` is the list of conversions applied on top; every step is also written
into `run['calibration']` so the file documents how its physical channels were made.

* Stresses: the lab's pressure-transducer calibration for a 1D fault with a 5 cm
  cross-section (Tseng 2026, NTU thesis, Appendix B.3.3 / Table B.1):
  `sigma_n = 4.507 V - 0.410` MPa on A5 and `V = 0.109 tau` on A6. The runs reproduce
  their nominal normal stress within 2 %.
* Slip: the stage calibration of the eddy-current sensors, `d (mm) = slope * V + intercept`,
  slopes -94.7, -96.6 and -95.4 um/V for the sensors on B1, B2, B3 (R² >= 0.9997, RMSE <= 10 um,
  travel up to 2 mm); the other five use the mean slope until they are calibrated. The jig
  intercepts are dropped and every `slip_k` is zeroed on the first 0.5 s of the run. The
  slope is negative (voltage falls as the gap opens), so slip accumulates positive.
  `displacement`, the field the Event Analyzer uses, is the mean of the eight slip channels
  by default (`EddySlip(displacement=...)` takes a single channel such as `'slip_5'` instead).
* The LVDT slope is still unknown; `lvdt` stays in volts.

`POSITIONS` places the sensors on the sample (mm; x along the fault from its left end,
y across the fault, z up). The table is stored with each recorder's `raw_data` and inherited by `slip`;
unknown entries stay NaN until filled in.

Layout of each run: `time`; one block per recorder (`elsys`, and `ni` for run5) holding the
raw-file reference and that recorder's voltages as the channel array `raw_data` (`data`
(channels x samples), `channels`, `unit`, `positions`); the physical scalars `normal_stress`,
`shear_stress`, `friction`, `displacement` at the top level; `slip` (channel array `slip_1..8`
in um with `source`, `slope_mm_per_v` and `positions`); `units`, `calibration`, `sources`.

`EVENT_FIELDS` / `EVENT_WINDOW_S` say which Elsys channels, and how much of each 2 MHz
trigger block, **Extract Events** copies into an event (the PZTs and the whole
0.1 s + 0.5 s block by default).


`NI_DECIMATION` is the block-mean factor for the run's time history (500 kHz → 2 kHz); the
full-rate record stays reachable through `run['ni']`.

In [ ]:
ELSYS_CHANNELS = {
    'A1': 'pzt_1', 'A2': 'pzt_2', 'A3': 'pzt_3', 'A4': 'pzt_4',
    'A5': 'pressure_1', 'A6': 'pressure_2',
    'A7': 'lvdt',
    'B1': 'eddy_1', 'B2': 'eddy_2', 'B3': 'eddy_3', 'B4': 'eddy_4',
    'B5': 'eddy_5', 'B6': 'eddy_6', 'B7': 'eddy_7', 'B8': 'eddy_8',
}

CALIBRATION = Calibration(
    *pressure_transducers('1D-5cm'),                      # normal_stress, shear_stress (MPa)
    EddySlip({'eddy_1': -0.094727676, 'eddy_2': -0.096627292, 'eddy_3': -0.095384964},
             zero_window_s=0.5, displacement='mean'),      # slip_1..8 and displacement (um)
    Friction(),
    note='pressure: Tseng 2026 Table B.1 (1D, 5 cm); eddy: stage calibration of B1-B3, mean slope elsewhere',
)

# Sensor positions on the sample (mm); None = unknown (stored as NaN). Fill in from the sensor layout.
POSITIONS = {
    'pzt_1': None, 'pzt_2': None, 'pzt_3': None, 'pzt_4': None,
    'eddy_1': None, 'eddy_2': None, 'eddy_3': None, 'eddy_4': None,
    'eddy_5': None, 'eddy_6': None, 'eddy_7': None, 'eddy_8': None,
}
POSITION_FRAME = 'sample frame: x along the fault from its left end, y across the fault, z up'

EVENT_FIELDS = ['pzt_1', 'pzt_2', 'pzt_3', 'pzt_4']   # Elsys channels copied into events
EVENT_WINDOW_S = (0.1, 0.5)                             # seconds before / after the event

NI_CHANNELS = {   # research log, [T207] NI-6451 DAQ Trial Run (2026-08-20)
    'ai0': 'pressure_1', 'ai1': 'pressure_2', 'ai2': 'lvdt',
    'ai3': 'eddy_1',      # 9 V dry cell in series as an offset trial; the zeroing removes it
    'ai4': 'eddy_2', 'ai5': 'eddy_3', 'ai6': 'eddy_4',
    'ai7': 'eddy_5', 'ai8': 'eddy_6', 'ai9': 'eddy_7', 'ai10': 'eddy_8',
}
NI_META = {'channel_notes': {'ai3': '9 V dry cell wired in series before the DAQ (offset trial)'},
           'input_range_v': 10.0}
NI_DECIMATION = 250            # 500 kHz -> 2 kHz
NI_EVENT_WINDOW_S = (0.1, 0.5)  # full-rate NI samples copied into each event (all NI channels)
ALIGNMENT_FIELD = 'eddy_5'     # NI channel with the clearest slip steps (ai7)

## Read the runs

Runs with an NI file take the NI clock as the run clock; the Elsys offset comes from the NI trigger sample and the first Elsys trigger block.

In [ ]:
runs = []
for p in TPC5_FILES:
    if run_of(p) in NI_FILES:
        runs.append(run_from_tpc5_ni(p, NI_FILES[run_of(p)], ELSYS_CHANNELS, NI_CHANNELS, OUTPUT_FILE.parent,
                                     CALIBRATION, ni_decimation=NI_DECIMATION, ni_meta=NI_META,
                                     elsys_event_fields=EVENT_FIELDS, elsys_event_window_s=EVENT_WINDOW_S,
                                     ni_event_window_s=NI_EVENT_WINDOW_S,
                                     positions=POSITIONS, position_frame=POSITION_FRAME))
    else:
        runs.append(run_from_tpc5(p, ELSYS_CHANNELS, OUTPUT_FILE.parent, CALIBRATION,
                                  event_fields=EVENT_FIELDS, event_window_s=EVENT_WINDOW_S,
                                  positions=POSITIONS, position_frame=POSITION_FRAME))
pd.DataFrame([{
    'name': r['name'], 'source': ' + '.join(r['sources'].values()),
    'sigma_n nominal (MPa)': r['normal_stress_level'], 'sigma_n measured (MPa)': float(np.nanmean(r['normal_stress'])),
    'samples': r['time'].size, 'duration (s)': float(r['time'][-1] - r['time'][0]),
    'trigger blocks': len(r['elsys']['blocks']['block']),
    'Elsys -> run offset (s)': r['elsys']['time_offset'], 'displacement end (um)': float(r['displacement'][-1]),
} for r in runs])

## Clock check for run5

For every Elsys trigger, the largest 20 ms change of one NI eddy-current channel within
±1 s of the shifted trigger time. Slip must follow the PZT trigger by tens of milliseconds,
never precede it; triggers without a step are PZT-only events.

In [ ]:
run5 = next(r for r in runs if 'ni' in r)
ni_src = open_source(run5['ni'], OUTPUT_FILE.parent, run5)
alignment = pd.DataFrame(slip_step_table(ni_src, run5['elsys']['blocks']['trigger_time_run'], ALIGNMENT_FIELD,
                                         decimation=NI_DECIMATION))
display(alignment)
lags = alignment['lag_ms'].dropna()
print(f"offset {run5['elsys']['time_offset']:.4f} s; slip steps for {lags.size} of {len(alignment)} triggers; "
      f"lag after the PZT trigger: {lags.min():.0f}..{lags.max():.0f} ms" if lags.size else 'no slip steps found')

## Overview

Stresses and the mean slip of every run; shaded spans are the Elsys trigger blocks on the run's clock.

In [ ]:
fig, axs = plt.subplots(len(runs), 1, figsize=(10, 2.2 * len(runs)), sharex=True)
for ax, run in zip(np.atleast_1d(axs), runs):
    ax.plot(run['time'], run['normal_stress'], lw=0.6, label='normal_stress (MPa)')
    ax.plot(run['time'], run['shear_stress'], lw=0.6, label='shear_stress (MPa)')
    ax2 = ax.twinx()
    ax2.plot(run['time'], run['displacement'], lw=0.6, color='C2')
    ax2.set_ylabel('displacement (um)', color='C2')
    blocks, dt0 = run['elsys']['blocks'], run['elsys']['time_offset']
    for a, b in zip(blocks['start'], blocks['end']):
        ax.axvspan(a + dt0, b + dt0, color='0.8', zorder=-10)
    ax.set_ylabel(f"{run['name']}  {run['normal_stress_level']:g} MPa" + ('  (NI)' if 'ni' in run else ''))
    ax.legend(loc='upper left', fontsize='small')
axs[-1].set_xlabel('time (s)')
fig.tight_layout()

## Run5: NI slip against Elsys triggers

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(run5['time'], get_channel(run5, ALIGNMENT_FIELD.replace('eddy', 'slip')), lw=0.6, label=f"{ALIGNMENT_FIELD.replace('eddy', 'slip')} (um, NI)")
for t in run5['elsys']['blocks']['trigger_time_run']:
    ax.axvline(t, color='k', ls=':', lw=0.8)
ax.set_xlabel('time (s, NI clock)'); ax.set_ylabel('um'); ax.legend(loc='upper left')
ax.set_title('run5: eddy-current slip vs Elsys PZT triggers (dotted) after the clock shift')

## Experiment record and save

In [ ]:
EXPERIMENT = experiment(
    't0207', runs,
    date='2026-08-20', operators=['朱展穎', '張德權'], temperature_C=21.0, humidity_percent=57.0, flow_rate_ml_min=8.0,
    acquisition=('Runs 1-4: Elsys TranAX ECR dual mode, block 1 continuous at 2 kHz, trigger blocks at 2 MHz '
                 '(0.1 s pre, 0.5 s post); A1-A4 PZT, A5-A6 pressure, A7 LVDT, B1-B8 eddy current. '
                 'Run 5: mechanical channels on a National Instruments USB-6451 at 500 kHz (ai0-ai10, '
                 'block means for the time history), PZT on the Elsys; run time axis is the NI clock.'),
    notes='Run sheet: after run2 block 20 the E02 slip channel is suspect.',
)

In [ ]:
dm = DataManager()
dm.data = EXPERIMENT
dm.save_file(OUTPUT_FILE)
print(f'{OUTPUT_FILE}  {OUTPUT_FILE.stat().st_size / 1e6:.1f} MB')

## Check the file the way the explorer loads it

In [ ]:
check = DataManager()
check.load_file(OUTPUT_FILE)
rows = []
for run in check.get_data('runs'):
    row = {'run': run['name'], 'samples': len(run['time']),
           'series': len(aligned_fields(run, len(run['time']), recurse=False)),
           'trigger blocks': len(run['elsys']['blocks']['block']),
           'displacement end (um)': float(run['displacement'][-1])}
    for key in run['sources']:
        row[f'{key} file found'] = (OUTPUT_FILE.parent / run[key]['filename']).exists()
    rows.append(row)
pd.DataFrame(rows)

## Next

Open `t0207.h5` in Labquake Explorer, right-click `runs/[i]/shear_stress` → **Pick Events**,
then `event_indices` → **Extract Events**: each event gets the run's channels around the pick,
`elsys.original` (PZTs from the 2 MHz trigger block that contains the event, shifted onto the
run clock) and, for run5, `ni.original` (all NI channels at 500 kHz).